In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
df=pd.read_csv("cleaned_data.csv")
print("Data shape:",df.shape)
print(df.head() )

Data shape: (2930, 76)
   Order        PID  MS SubClass MS Zoning  Lot Frontage  Lot Area Street  \
0      1  526301100           20        RL         141.0     31770   Pave   
1      2  526350040           20        RH          80.0     11622   Pave   
2      3  526351010           20        RL          81.0     14267   Pave   
3      4  526353030           20        RL          93.0     11160   Pave   
4      5  527105010           60        RL          74.0     13830   Pave   

  Lot Shape Land Contour Utilities  ... Enclosed Porch 3Ssn Porch  \
0       IR1          Lvl    AllPub  ...              0          0   
1       Reg          Lvl    AllPub  ...              0          0   
2       IR1          Lvl    AllPub  ...              0          0   
3       Reg          Lvl    AllPub  ...              0          0   
4       IR1          Lvl    AllPub  ...              0          0   

  Screen Porch Pool Area Misc Val Mo Sold Yr Sold  Sale Type  Sale Condition  \
0            0     

In [2]:
y_reg=df["SalePrice"]
y_clf=(df["SalePrice"] > df["SalePrice"].median()).astype(int)
X=df.drop(columns=["SalePrice"])
print("Feature shape:",X.shape)
print("Regression target shape:",y_reg.shape)
print("Classification target shape:",y_clf.shape)
print("class distribution:",y_clf.value_counts())

Feature shape: (2930, 75)
Regression target shape: (2930,)
Classification target shape: (2930,)
class distribution: SalePrice
0    1467
1    1463
Name: count, dtype: int64


In [3]:
categorical_columns=X.select_dtypes(include=["object","category"]).columns
print("Categorical columns:",categorical_columns)
X=pd.get_dummies(X,columns=categorical_columns,drop_first=True)
print("After encoding shape:",X.shape)

Categorical columns: Index(['MS Zoning', 'Street', 'Lot Shape', 'Land Contour', 'Utilities',
       'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl',
       'Exterior 1st', 'Exterior 2nd', 'Exter Qual', 'Exter Cond',
       'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC',
       'Central Air', 'Electrical', 'Kitchen Qual', 'Functional',
       'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond',
       'Paved Drive', 'Sale Type', 'Sale Condition'],
      dtype='object')
After encoding shape: (2930, 244)


In [4]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_clf_train,y_clf_test=train_test_split(X,y_clf,test_size=0.2,random_state=42,stratify=y_clf)
print("Training feature:",X_train.shape)
print("Testing feature:",X_test.shape)

Training feature: (2344, 244)
Testing feature: (586, 244)


In [5]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
print("Training data shape:",X_train_scaled.shape)
print("Testing data shape:",X_test_scaled.shape)

Training data shape: (2344, 244)
Testing data shape: (586, 244)


In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
dt_clf=DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train_scaled,y_clf_train)
train_pred=dt_clf.predict(X_train_scaled)
test_pred=dt_clf.predict(X_test_scaled)
train_acc=accuracy_score(y_clf_train,train_pred)
test_acc=accuracy_score(y_clf_test,test_pred)
print("Default Decision Tree")
print("Training accuracy:",train_acc)
print("Testing accuracy:",test_acc)

Default Decision Tree
Training accuracy: 1.0
Testing accuracy: 0.8668941979522184


In [7]:
dt_controlled=DecisionTreeClassifier(max_depth=5,min_samples_split=20,random_state=42)
dt_controlled.fit(X_train_scaled,y_clf_train)
train_pred_controlled=dt_controlled.predict(X_train_scaled)
test_pred_controlled=dt_controlled.predict(X_test_scaled)
train_acc_controlled=accuracy_score(y_clf_train,train_pred_controlled)
test_acc_controlled=accuracy_score(y_clf_test,test_pred_controlled)
print("Controlled Decision Tree")
print("Training accuracy:",train_acc_controlled)
print("Testing accuracy:",test_acc_controlled)

Controlled Decision Tree
Training accuracy: 0.9261945392491467
Testing accuracy: 0.8993174061433447


In [8]:
dt_gini=DecisionTreeClassifier(criterion="gini",max_depth=5,random_state=42)
dt_gini.fit(X_train_scaled,y_clf_train)
gini_pred=dt_gini.predict(X_test_scaled)
gini_acc=accuracy_score(y_clf_test,gini_pred)
dt_entropy=DecisionTreeClassifier(criterion="entropy",max_depth=5,random_state=42)
dt_entropy.fit(X_train_scaled,y_clf_train)
entropy_pred=dt_entropy.predict(X_test_scaled)
entropy_acc=accuracy_score(y_clf_test,entropy_pred)
print("Gini test accuracy:",gini_acc)
print("Entropy test accuracy:",entropy_acc)

Gini test accuracy: 0.9010238907849829
Entropy test accuracy: 0.909556313993174


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
rf_model=RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42)
rf_model.fit(X_train_scaled,y_clf_train)
train_predict=rf_model.predict(X_train_scaled)
test_predict=rf_model.predict(X_test_scaled)
test_prob=rf_model.predict_proba(X_test_scaled)[:,1]
train_accuracy=accuracy_score(y_clf_train,train_predict)
test_accuracy=accuracy_score(y_clf_test,test_predict)
roc_auc=roc_auc_score(y_clf_test,test_prob)
print("Random Forest")
print("Training accuracy:",train_accuracy)
print("Testing accuracy:",test_accuracy)
print("Roc-Auc:",roc_auc)
feature_importance=pd.DataFrame({"Feature":X.columns,"Importance":rf_model.feature_importances_})
top5=feature_importance.sort_values(by="Importance",ascending=False).head(5)
print("Top 5 Features")
print(top5)

Random Forest
Training accuracy: 0.9893344709897611
Testing accuracy: 0.9300341296928327
Roc-Auc: 0.9799764703141562
Top 5 Features
          Feature  Importance
20      Full Bath    0.088863
5    Overall Qual    0.070389
17    Gr Liv Area    0.056855
7      Year Built    0.052555
26  Garage Yr Blt    0.040278


In [10]:
from sklearn.ensemble import GradientBoostingClassifier
gb_model=GradientBoostingClassifier(n_estimators=100,learning_rate=0.1,max_depth=3,random_state=42)
gb_model.fit(X_train_scaled,y_clf_train)
gb_train_predict=gb_model.predict(X_train_scaled)
gb_test_predict=gb_model.predict(X_test_scaled)
gb_test_prob=gb_model.predict_proba(X_test_scaled)[:,1]
gb_train_accuracy=accuracy_score(y_clf_train,gb_train_predict)
gb_test_accuracy=accuracy_score(y_clf_test,gb_test_predict)
gb_roc_auc=roc_auc_score(y_clf_test,gb_test_prob)
print("Gradient Boosting")
print("Training accuracy:",gb_train_accuracy)
print("Testing accuracy:",gb_test_accuracy)
print("Roc-Auc:",gb_roc_auc)

Gradient Boosting
Training accuracy: 0.9765358361774744
Testing accuracy: 0.9266211604095563
Roc-Auc: 0.9834010879567613


In [11]:
least5=feature_importance.sort_values(by="Importance",ascending=True).head(5)
print("Least 5 Features")
print(least5)
least_features=least5["Feature"].tolist()
X_train_reduced=X_train.drop(columns=least_features)
X_test_reduced=X_test.drop(columns=least_features)
scaler_reduced=StandardScaler()
X_train_reduced_scaled=scaler_reduced.fit_transform(X_train_reduced)
X_test_reduced_scaled=scaler_reduced.transform(X_test_reduced)
rf_reduced=RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42)
rf_reduced.fit(X_train_reduced_scaled,y_clf_train)
rf_reduced_prob=rf_reduced.predict_proba(X_test_reduced_scaled)[:,1]
rf_reduced_auc=roc_auc_score(y_clf_test,rf_reduced_prob)
print("Full model roc-auc:",roc_auc)
print("Reduced model roc-auc:",rf_reduced_auc)

Least 5 Features
                 Feature  Importance
60   Neighborhood_BrDale         0.0
40     MS Zoning_I (all)         0.0
99      Condition 2_RRAn         0.0
100     Condition 2_RRNn         0.0
120       Roof Matl_Roll         0.0
Full model roc-auc: 0.9799764703141562
Reduced model roc-auc: 0.9803259210940138


In [12]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
lr_model=LogisticRegression(max_iter=1000,random_state=42)
models={"LogisticRegression":lr_model,"Controlled Decision Tree":dt_controlled,"Random Forest":rf_model,"Gradient Boosting":gb_model}
results=[]
for name,model in models.items():
  scores=cross_val_score(model,X_train_scaled,y_clf_train,cv=cv,scoring="roc_auc")
  results.append({"Model":name,"Mean Auc":scores.mean(),"Std Auc":scores.std()})
cv_results=pd.DataFrame(results)
print("5 Fold cross validation result")
print(cv_results)

5 Fold cross validation result
                      Model  Mean Auc   Std Auc
0        LogisticRegression  0.968428  0.004193
1  Controlled Decision Tree  0.938063  0.008320
2             Random Forest  0.982310  0.001801
3         Gradient Boosting  0.981890  0.004460


In [13]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
pipeline=make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),RandomForestClassifier(random_state=42))
param_grid={"randomforestclassifier__n_estimators":[50,100,200],"randomforestclassifier__max_depth":[5,10,None],"randomforestclassifier__min_samples_leaf":[1,5]}
grid_search=GridSearchCV(estimator=pipeline,param_grid=param_grid,cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42),scoring="roc_auc",n_jobs=-1)
grid_search.fit(X_train,y_clf_train)
print("Best parameters:",grid_search.best_params_)
print("Best cross validation auc:",grid_search.best_score_)
best_pipeline=grid_search.best_estimator_

Best parameters: {'randomforestclassifier__max_depth': 10, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 200}
Best cross validation auc: 0.9825470209812817


In [14]:
fractions=[0.2,0.4,0.6,0.8,1.0]
learning_results=[]
for f in fractions:
  size=int(f*len(X_train))
  X_train_subset=X_train.iloc[:size]
  y_train_subset=y_clf_train.iloc[:size]
  best_pipeline.fit(X_train_subset,y_train_subset)
  train_prob=best_pipeline.predict_proba(X_train_subset)[:,1]
  train_auc=roc_auc_score(y_train_subset,train_prob)
  test_prob=best_pipeline.predict_proba(X_test)[:,1]
  test_auc=roc_auc_score(y_clf_test,test_prob)
  learning_results.append({"Training Fraction":f,"Training Auc":train_auc,"Test Auc":test_auc})
learning_curve=pd.DataFrame(learning_results)
print("Learning curve")
print(learning_curve)

Learning curve
   Training Fraction  Training Auc  Test Auc
0                0.2      1.000000  0.972341
1                0.4      0.999982  0.976365
2                0.6      0.999808  0.980105
3                0.8      0.999634  0.981025
4                1.0      0.999393  0.979441


In [15]:
import joblib
joblib.dump(best_pipeline,"best_model.pkl")
print("best_model.pkl saved successfully!")

best_model.pkl saved successfully!


In [16]:
loaded_model=joblib.load("best_model.pkl")
sample_data=X_test.iloc[:2]
predictions=loaded_model.predict(sample_data)
print("Sample prediction:",predictions)

Sample prediction: [1 1]
